# Большая шпаргалка по визуализации: Matplotlib и Seaborn

Мы прошли pandas, теперь учимся рисовать. В Python две главные библиотеки:

1.  **Matplotlib** — фундамент. Позволяет контролировать каждый штрих, но требует много кода. Используется, когда нужен сложный нестандартный график или настройка осей.
2.  **Seaborn** — надстройка. Делает красиво и статистически правильно одной строкой. Отлично работает с DataFrame.

Мы пойдем по принципу сравнения: **Как это сделать в «чистом» Matplotlib** vs **Как это сделать в Seaborn**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Магия Jupyter для отображения графиков
%matplotlib inline

# Создадим насыщенный датасет для примеров
# Представим, что это статистика сотрудников IT-компании
data = {
    'Отдел': ['IT']*50 + ['HR']*30 + ['Sales']*40,
    'Зарплата': np.concatenate([np.random.normal(200, 40, 50), np.random.normal(100, 15, 30), np.random.normal(150, 50, 40)]),
    'Возраст': np.random.randint(20, 60, 120),
    'Опыт (лет)': np.random.randint(1, 20, 120),
    'Удовлетворенность': np.random.randint(1, 10, 120),
    'Удаленка': np.random.choice(['Да', 'Нет'], 120)
}

df = pd.DataFrame(data)
df.head()

---

## 1. Гистограмма (Histogram)
**Зачем:** Показать распределение одной величины

In [ ]:
# MATPLOTLIB
plt.figure(figsize=(8, 4))
plt.hist(df['Зарплата'], bins=20, color='gray', edgecolor='black')
plt.title('Matplotlib: Распределение зарплат')
plt.show()

# SEABORN
plt.figure(figsize=(8, 4))
# kde=True рисует плавную линию плотности вероятности
sns.histplot(df['Зарплата'], bins=20, kde=True, color='purple')
plt.title('Seaborn: Распределение зарплат (с линией)')
plt.show()

---

## 2. Boxplot («Ящик с усами»)
**Важнейший график для аналитика.** 
Показывает медиану, квартили (25% и 75%) и **выбросы** (точки за пределами усов).

In [ ]:
# MATPLOTLIB
# Требует подготовки данных: нужно вручную разбить на списки по группам
salaries_it = df[df['Отдел']=='IT']['Зарплата']
salaries_hr = df[df['Отдел']=='HR']['Зарплата']

plt.figure(figsize=(6, 4))
plt.boxplot([salaries_it, salaries_hr], labels=['IT', 'HR'])
plt.title('Matplotlib Boxplot')
plt.show()

# SEABORN
# Делает всю группировку сам параметрами x и y
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Отдел', y='Зарплата', palette='Set2')
plt.title('Seaborn Boxplot: Кто сколько зарабатывает')
plt.show()

---

## 3. Violin Plot (Скрипичный ключ)
**Зачем:** Это гибрид Boxplot и Гистограммы. 
Если Boxplot показывает только статистику, то Violin показывает форму распределения («пузатость»). Видно, где именно скучковались люди.

In [ ]:
plt.figure(figsize=(8, 5))
# split=True позволяет сравнить две подгруппы (Удаленка) внутри одной "скрипки"
sns.violinplot(data=df, x='Отдел', y='Зарплата', hue='Удаленка', split=True, palette='Pastel1')
plt.title('Violin Plot: Распределение зарплат с учетом удаленки')
plt.show()

---

## 4. Столбчатая диаграмма (Bar Plot)
**Зачем:** Сравнить средние (или суммы) по категориям.

*Внимание:* Seaborn по умолчанию считает **среднее** и рисует черную палочку сверху — это доверительный интервал (ошибка измерения).

In [ ]:
# MATPLOTLIB
# Нужно сначала сгруппировать данные pandas-ом!
mean_vals = df.groupby('Отдел')['Удовлетворенность'].mean()

plt.figure(figsize=(6, 4))
plt.bar(mean_vals.index, mean_vals.values, color='orange')
plt.title('Matplotlib Bar: Средняя удовлетворенность')
plt.show()

# SEABORN
# Считает агрегацию сам
plt.figure(figsize=(6, 4))
sns.barplot(data=df, x='Отдел', y='Удовлетворенность', palette='magma')
plt.title('Seaborn Bar: Средняя удовлетворенность')
plt.show()

---

## 5. Scatter Plot (Точечная диаграмма)
**Зачем:** Ищем корреляцию. Зависит ли X от Y?

In [ ]:
# MATPLOTLIB
plt.figure(figsize=(6, 4))
plt.scatter(df['Опыт (лет)'], df['Зарплата'], alpha=0.5) # alpha - прозрачность
plt.title('Matplotlib: Опыт vs Зарплата')
plt.show()

# SEABORN
plt.figure(figsize=(8, 5))
# size - размер точек зависит от возраста
# hue - цвет зависит от отдела
sns.scatterplot(data=df, x='Опыт (лет)', y='Зарплата', hue='Отдел', size='Возраст', sizes=(20, 200))
plt.legend(bbox_to_anchor=(1.05, 1), loc=2) # Выносим легенду за график
plt.title('Seaborn: Мульти-факторный анализ')
plt.show()

---

## 6. Heatmap (Тепловая карта)
**Зачем:** Смотреть матрицу корреляций. Что с чем связано сильнее всего?

In [ ]:
# Сначала считаем матрицу корреляций в Pandas
corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', linewidths=1, vmin=-1, vmax=1)
plt.title('Матрица корреляций')
plt.show()

---

## 7. PRO-уровень Matplotlib: Субплоты (Subplots)

Часто нужно нарисовать несколько графиков рядом. Для этого используется "ООП-стиль" Matplotlib.
Мы создаем `figure` (холст) и массив `axes` (осей/графиков).

Синтаксис: `fig, ax = plt.subplots(строки, столбцы)`

In [ ]:
# Создаем холст с 1 строкой и 2 колонками
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График 1: Рисуем в левой ячейке (axes[0])
sns.histplot(df['Зарплата'], ax=axes[0], color='teal')
axes[0].set_title('Гистограмма зарплат')

# График 2: Рисуем в правой ячейке (axes[1])
sns.boxplot(data=df, x='Отдел', y='Зарплата', ax=axes[1])
axes[1].set_title('Боксплот зарплат')

plt.show()

## Что еще бывает?

В библиотеках сотни графиков, не стесняйтесь гуглить галерею:
1.  **Pairplot:** Строит графики всех пар колонок друг с другом.
2.  **Jointplot:** Scatterplot в центре + гистограммы по бокам.
3.  **Pie Chart:** Круговая диаграмма (лучше не используйте, бары лучше).


Документация:
- [Matplotlib Gallery](https://matplotlib.org/stable/gallery/index.html)
- [Seaborn Gallery](https://seaborn.pydata.org/examples/index.html)